In [ ]:
# import pandas as pd
# import requests
# import time
# import os

# API_KEY = "LAST_FM_KEY"

# INPUT_FILE = "song_main_artist.csv"
# CHECKPOINT_FILE = "song_main_artist_checkpoint.csv"
# FINAL_FILE = "song_main_artist_with_genres.csv"

# TRACK_COLUMN = "name"
# ARTIST_COLUMN = "main_artist"
# OUTPUT_COLUMN = "genres"

# CHECKPOINT_INTERVAL = 100

### CONFIGURATION 
This cell inlcudes the basic configs like the last.fm api key  
name of the input file with all the records  
a checkpoint file to which data is inserted to after every 100 calls  
and a final file in which all the record is inserted to.  
Then there is the track_column (song name), artist_column(name of artist) and output_column(all the genre tags)  
checkpoint interval which species amount of reads after which data is added to checkpoint file  
list of bad tags which is used to remove noise tags

In [ ]:
import pandas as pd
import requests
import time
import os
import re

API_KEY = "LAST_FM_KEY"

INPUT_FILE = "songs_main_artist.csv"
CHECKPOINT_FILE = "songs_main_artist_checkpoint.csv"
FINAL_FILE = "songs_main_artist_with_genres.csv"

TRACK_COLUMN = "name"
ARTIST_COLUMN = "main_artist"
OUTPUT_COLUMN = "genres"

CHECKPOINT_INTERVAL = 100

BAD_TAGS = [
    "seen live", "favorites", "favorite",
    "female vocalists", "male vocalists",
    "90s", "2000s", "2010s",
    "british", "american"
]
BAD_TAGS = set(tag.lower().strip() for tag in BAD_TAGS)

## Tag Normalization Mapping

This mapping standardizes different genre labels into a consistent format for better processing and recommendation quality.


In [ ]:
TAG_MAPPING = {
    "hip hop": "hip-hop",
    "hiphop": "hip-hop",
    "rap": "hip-hop",

    "rnb": "r&b",

    "alt rock": "alternative rock",

    "electro": "electronic",
    "electronica": "electronic"
}

def normalize_tag(tag):
    tag = tag.lower().strip()
    tag = re.sub(r"\s+", " ", tag)
    return TAG_MAPPING.get(tag, tag)


## Fetching Tags from Last.fm API

This section retrieves genre tags for both tracks and artists using the Last.fm API. It is designed to be robust and efficient, especially when processing large datasets.

---

### Track Tag Retrieval

- Fetches top tags for each track using its name and artist  
- Uses multiple retry attempts to handle temporary network or API failures  

---

### Artist Tag Retrieval (Fallback)

- If track-level tags are unavailable, artist-level tags are used as a fallback  
- Ensures better coverage when track data is sparse or missing  
- Also includes retry logic for reliability  

---

### Checkpointing & Resume Mechanism

- Supports resuming from a saved checkpoint file  
- Prevents reprocessing of already completed entries  
- If no checkpoint exists, processing starts from scratch  
- Automatically initializes the output column if missing  

---

### Key Features

- Fault-tolerant with retry handling  
- Efficient for large-scale data collection  
- Uses fallback strategy to improve tag availability  
- Maintains progress using checkpoint files  

---

In [ ]:
def fetch_track_tags(track, artist, retries=3):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "track.getTopTags",
        "track": track,
        "artist": artist,
        "api_key": API_KEY,
        "format": "json"
    }
    
    for attempt in range(retries):
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
            tags = data.get("toptags", {}).get("tag", [])
            if isinstance(tags, list):
                return [(t["name"], int(t.get("count", 0))) for t in tags if "name" in t]
            return []
        except Exception:
            print(f"⚠️ Track retry {attempt+1} failed: {track} - {artist}")
            time.sleep(1)
    return []

def fetch_artist_tags(artist, retries=3):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getTopTags",
        "artist": artist,
        "api_key": API_KEY,
        "format": "json"
    }

    for attempt in range(retries):
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
            tags = data.get("toptags", {}).get("tag", [])
            if isinstance(tags, list):
                return [(t["name"], int(t.get("count", 0))) for t in tags if "name" in t]
            return []
        except Exception:
            print(f"⚠️ Artist retry {attempt+1} failed: {artist}")
            time.sleep(1)
    return []

if os.path.exists(CHECKPOINT_FILE):
    print("🔁 Resuming from checkpoint...")
    df = pd.read_csv(CHECKPOINT_FILE)
else:
    print("🆕 Starting fresh...")
    df = pd.read_csv(INPUT_FILE)

    if OUTPUT_COLUMN not in df.columns:
        df[OUTPUT_COLUMN] = None

print("Columns:", df.columns.tolist())


## Resume Index Detection

This step determines where to continue processing in the dataset.

---

### Logic Overview

- Checks which rows have already been processed  
- Identifies the last completed index  
- Sets the starting point for the next run accordingly  

---

### How It Works

- If processed entries exist:
  - Finds the last processed row  
  - Starts from the next index  

- If no entries are processed:
  - Starts from the beginning (index 0)  

---

### Purpose

- Enables seamless continuation after interruptions  
- Avoids redundant API calls and computations  
- Improves efficiency for large datasets  

---

### Output

- Prints the index from which processing will resume  

In [ ]:
processed_mask = df[OUTPUT_COLUMN].notna()

if processed_mask.any():
    start_idx = processed_mask[processed_mask].index.max() + 1
else:
    start_idx = 0

print(f"▶️ Starting from index: {start_idx}")


## Tag Extraction and Processing Loop

This section iterates through the dataset to fetch, clean, and store genre tags for each track.

---

### Processing Workflow

- Iterates over rows starting from the computed resume index  
- Extracts track name and artist for each entry  
- Skips rows with missing or invalid values  

---

### Track Tag Handling

- Retrieves tags using track-level API  
- Normalizes each tag for consistency  
- Filters out:
  - Irrelevant or predefined unwanted tags  
  - Tags matching the artist name  
- Removes duplicates while preserving order  

---

### Fallback Strategy

- If no valid track tags are found:
  - Fetches tags using artist-level API  
  - Applies the same normalization and filtering steps  
  - Sorts tags by frequency  
  - Selects the top 3 most relevant tags  

---

### Output Assignment

- Stores the final cleaned tags as a comma-separated string  
- Associates tags with the corresponding dataset row  

---

### Progress Control

- Adds a short delay between API calls to avoid rate limits  
- Logs progress for each processed track  

---

### Checkpointing

- Saves intermediate results at fixed intervals  
- Prevents data loss during long executions  
- Allows safe resumption from the last saved state  

---

### Final Output

- Writes the fully processed dataset to the final output file  
- Ensures all rows contain cleaned and standardized tags  

In [3]:
for idx in range(start_idx, len(df)):

    track = str(df.at[idx, TRACK_COLUMN]).strip()
    artist = str(df.at[idx, ARTIST_COLUMN]).strip()

    if not track or track == "nan" or not artist or artist == "nan":
        continue

    print(f"\n🎵 {idx}: {track} - {artist}")

    raw_tags = fetch_track_tags(track, artist)
    source = "track"

    artist_norm = artist.lower().strip()

    cleaned_tags = []

    for tag, count in raw_tags:
        tag_norm = normalize_tag(tag)

        if tag_norm in BAD_TAGS:
            continue
        if tag_norm == artist_norm:
            continue

        cleaned_tags.append(tag_norm)

    cleaned_tags = list(dict.fromkeys(cleaned_tags))

    if not cleaned_tags:
        print("   ⚠️ No usable track tags → using artist fallback")
        raw_artist_tags = fetch_artist_tags(artist)
        source = "artist"

        processed = []

        for tag, count in raw_artist_tags:
            tag_norm = normalize_tag(tag)

            if tag_norm in BAD_TAGS:
                continue
            if tag_norm == artist_norm:
                continue

            processed.append((tag_norm, count))

        # sort by count
        processed.sort(key=lambda x: x[1], reverse=True)

        # take top 3
        cleaned_tags = [tag for tag, _ in processed[:3]]

        cleaned_tags = list(dict.fromkeys(cleaned_tags))

    if cleaned_tags:
        print(f"   → ({source}) Genres: {', '.join(cleaned_tags)}")
    else:
        print("   → Genres: NONE (after filtering)")

    df.at[idx, OUTPUT_COLUMN] = ",".join(cleaned_tags)

    time.sleep(0.3)

    if idx % CHECKPOINT_INTERVAL == 0 and idx != 0:
        df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"💾 Checkpoint saved at index {idx}")

df.to_csv(FINAL_FILE, index=False)
print("\n✅ Finished! File saved as:", FINAL_FILE)

🔁 Resuming from checkpoint...
Columns: ['name', 'main_artist', 'genres']
▶️ Starting from index: 11901

🎵 11901: High Horse - Kacey Musgraves
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: country, pop, singer-songwriter

🎵 11902: Same Bitches (feat. G-Eazy & YG) - Post Malone
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: hip-hop, trap

🎵 11903: MIC Drop (Steve Aoki Remix) [Full Length Edition] - BTS
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: k-pop, kpop, korean

🎵 11904: Falling (blackbear Remix) - Trevor Daniel
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: pop, r&b, hip-hop

🎵 11905: Purity (feat. Frank Ocean) - A$AP Rocky
   → (track) Genres: soul, hip-hop, asap rocky, frank ocean

🎵 11906: Girl Like You - Jason Aldean
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: country, modern country, male country

🎵 11907: MONSTERS - Shinedown
   ⚠️ No usable 

KeyboardInterrupt: 